In [ ]:
import pandas as pd
import os

# ================= 1. 配置区 =================
files = {
    "DeepSeek": "output/DeepSeek Results_Sorted.csv",
    "Qwen": "output/Qwen Results_Sorted.csv",
    "Kimi": "output/Kimi Results_Sorted.csv"
}

# 定义你文档中出现的所有品牌，用来检测 AI 到底推荐了谁
BRANDS = ["Champion", "Nike", "Adidas", "L'Oreal", "欧莱雅", "Lancôme", "兰蔻", "Chanel", "香奈儿", "OPPO", "Xiaomi", "小米", "Apple", "苹果"]

# ================= 2. 分析函数 =================
def analyze_file(model_name, file_path):
    if not os.path.exists(file_path):
        print(f"⚠️ 找不到 {model_name} 的文件。")
        return None
    
    df = pd.read_csv(file_path, encoding='utf-8-sig')
    
    # 指标 1：检测推荐了哪个品牌 (打标)
    def check_brand(text):
        if not isinstance(text, str): return "None"
        found_brands = [b for b in BRANDS if b.lower() in text.lower()]
        return found_brands[0] if found_brands else "Others/None"
    
    df['Recommended_Brand'] = df['Response'].apply(check_brand)
    
    # 指标 2：计算字数
    df['Word_Count'] = df['Response'].apply(lambda x: len(str(x)))
    
    # 分组统计：按 品类、格式、导购类型 聚合，计算品牌的被推荐概率
    # 这里以“只要不是 None，就代表推荐成功”为例
    df['Is_Recommended'] = df['Recommended_Brand'].apply(lambda x: 0 if x == "Others/None" else 1)
    
    summary = df.groupby(['Category', 'Format', 'Guide_Type'])['Is_Recommended'].mean() * 100
    summary = summary.reset_index()
    summary.rename(columns={'Is_Recommended': 'Recommendation_Rate(%)'}, inplace=True)
    summary['Model'] = model_name
    
    return summary

# ================= 3. 执行并合并结果 =================
all_summaries = []
for model, path in files.items():
    res = analyze_file(model, path)
    if res is not None:
        all_summaries.append(res)

if all_summaries:
    final_summary = pd.concat(all_summaries, ignore_index=True)
    # 保存结果
    final_summary.to_csv("output/GEO_Analysis_Summary.csv", index=False, encoding='utf-8-sig')
    print("✅ 分析完成！汇总报表已保存至GEO_Analysis_Summary.csv")
    print(final_summary.head(27)) # 打印前 27 行看看

✅ 分析完成！汇总报表已保存至GEO_Analysis_Summary.csv
       Category     Format  Guide_Type  Recommendation_Rate(%)     Model
0       Apparel        FAQ         1.1                   100.0  DeepSeek
1       Apparel        FAQ         1.2                   100.0  DeepSeek
2       Apparel        FAQ         1.3                   100.0  DeepSeek
3       Apparel       List         1.1                    97.0  DeepSeek
4       Apparel       List         1.2                    85.0  DeepSeek
5       Apparel       List         1.3                   100.0  DeepSeek
6       Apparel  Paragraph         1.1                   100.0  DeepSeek
7       Apparel  Paragraph         1.2                    94.5  DeepSeek
8       Apparel  Paragraph         1.3                   100.0  DeepSeek
9     Cosmetics        FAQ         1.1                   100.0  DeepSeek
10    Cosmetics        FAQ         1.2                   100.0  DeepSeek
11    Cosmetics        FAQ         1.3                   100.0  DeepSeek
12    Cosme